In [76]:
%pip install venv

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement venv (from versions: none)
ERROR: No matching distribution found for venv


In [ ]:
from bs4 import BeautifulSoup
import os
import pandas as pd
from urllib.parse import urljoin
import requests
import re

base_url = "https://books.toscrape.com/catalogue/"

os.makedirs("./books_scrapper/data/raw", exist_ok=True)
os.makedirs("./books_scrapper/images/raw", exist_ok=True)

books = []
def get_rating(star_rating):
    return ["zero", "one", "two", "three", "four", "five"].index(star_rating.lower())

url = urljoin(base_url, "page-1.html")

page_no = 1

while url and page_no <= 3:
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    articles = soup.find_all("article", class_="product_pod")
    
    for article in articles:
        title = article.h3.a["title"]
        price = article.find("p", class_="price_color").text[2:]
        rating_class = article.find("p", class_="star-rating")["class"][1]
        rating = get_rating(rating_class)
        img_url = urljoin(base_url, article.find("img")["src"])
        img_name = img_url.split("/")[-1]
        img_response = requests.get(img_url).content
        image_name = f"{title.split(':')[0][:50].strip()}.jpg"
        with open(f"books_scrapper/images/raw/{img_name}", "wb") as img_file:
            img_file.write(img_response)
        image_path = os.rename(f"books_scrapper/images/raw/{img_name}", f"books_scrapper/images/raw/{image_name}")
        
        try:
            books.append({
                "title": title,
                "price": float(re.sub(r"[^0-9.]", "", price)),
                "rating": rating,
                "image": img_response,
                "image_name": image_name
            })
            print(f"Scraped: {title}")
        except Exception as e:
            print(f"Error occurred while processing article: {e}")
            continue
    print(f"Completed page {page_no}")
    next_button = soup.find("li", class_="next")
    url = urljoin(base_url, next_button.a["href"]) if next_button else None
    page_no += 1
    url = urljoin(base_url, f"page-{page_no}.html")
    
df = pd.DataFrame(books)
df.to_csv("books_scrapper/data/raw/books.csv", index=False)
print("Scraping completed. Data saved in 'books_scrapper/data/raw/books.csv' and images saved in 'books_scrapper/images/raw' directory.")
        

In [ ]:
df.image_name

In [ ]:
import pandas as pd
import os

os.makedirs("books_scrapper/data/processed", exist_ok=True)
os.makedirs("books_scrapper/images/processed", exist_ok=True)



df = pd.read_csv("books_scrapper/data/raw/books.csv")

df = df.drop_duplicates()
df["price"] = df["price"].fillna(df["price"].mean())

summary = df.groupby("rating")["price"].agg(["count", "mean"])

df.to_csv("books_scrapper/data/processed/cleaned_books.csv", index=False)
summary.to_csv("books_scrapper/data/processed/summary.csv")

print("Data processing completed. Cleaned data and summary saved in 'books_scrapper/data/processed/' directory.")


image_path = "books_scrapper/images/raw/"
new_image_path = "books_scrapper/images/processed/"

if os.path.exists(image_path) and len(os.listdir(image_path)) > 0:
    images = os.listdir(image_path)
    print(f"Found {len(images)} images in '{image_path}' directory.")
    for image in images:
        old_path = os.path.join(image_path, image)
        new_path = os.path.join(new_image_path, image)
        os.rename(old_path, new_path)
        
    print(f"Images moved to {new_image_path} directory.")
elif os.path.exists(image_path) and len(os.listdir(image_path)) == 0:
    print(f"No images found in '{image_path}' directory.")
    
else:
    print(f"Directory '{image_path}' does not exist.")

In [ ]:
df.info()

In [ ]:
import pandas as pd
import os
import shutil

cleaned_books_path = "./data/processed/cleaned_books.csv"
if os.path.exists(cleaned_books_path):
    df = pd.read_csv(cleaned_books_path)
else:
    print(f"File not found: {cleaned_books_path}")
    df = pd.DataFrame()  # create empty DataFrame as fallback

In [ ]:

summary_path = "./data/processed/summary.csv"
if os.path.exists(summary_path):
    summary = pd.read_csv(summary_path)
else:
    print(f"File not found: {summary_path}")
    summary = pd.DataFrame()  # create empty DataFrame as fallback


In [ ]:
df.info()

In [ ]:

base_target_folder = '../images/'
source_folder = './images/processed/'

for idx, row in df.iterrows():
    image = row['image']
    rate = int(row['rating'])  # Use the value from the current row, not the whole column
    image_name = row['image_name']  # If your image column contains the filename
    src_path = os.path.join(source_folder, image_name)
    target_folder = os.path.join(base_target_folder, f'{rate}_star')
    os.makedirs(target_folder, exist_ok=True)
    dst_path = os.path.join(target_folder, image_name)
    if os.path.exists(src_path):
        shutil.move(src_path, dst_path)
        print(f"Moved {image_name} to {target_folder}")
    else:
        print(f"File not found: {src_path}")